In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import os
import pandas as pd
import glob
import cv2
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Basic function
def mean(array):
    return np.mean(np.array(array).flatten())
def mode(array):
    return np.bincount(np.array(array).flatten()).argmax()
def min1(array):
    return np.min(np.array(array).flatten())
def max1(array):
    return np.max(np.array(array).flatten())
def std1(array):
    return np.std(np.array(array).flatten())
def rgb(indir):
    # get values
    RGB_values=[]
    for image_location in glob.glob(indir):
        concentration = image_location.split("/")[-1].split('-')[0]
        image_id=image_location.split("/")[-1]
        # load
        image = pd.read_csv(image_location)
        RGB_values.append([image_id,concentration,image["B"].mean(),image["G"].mean(),image["R"].mean(),
                                                  image["B"].mode()[0],image["G"].mode()[0],image["R"].mode()[0],
                                                  image["B"].min(),image["G"].min(),image["R"].min(),
                                                  image["B"].max(),image["G"].max(),image["R"].max(),
                                                  image["B"].std(),image["G"].std(),image["R"].std()
                        ])
    columns_name=["image","concentration",
    "meanB","meanG","meanR",
    "modeB","modeG","modeR",
    "minB","minG","minR",
    "maxB","maxG","maxR",
    "stdB","stdG","stdR"]

    df = pd.DataFrame(RGB_values, columns=columns_name)
    df["batch"]=df["image"].apply(lambda x: x.split(".csv")[0].split("_")[1])
    df["concentration"]=df["concentration"].astype(float)
    # get standard deviation 
    # note: this is different with std for each figures, this std for all figures with the same scene (1 concentration=10 figures for ex)
    std_result=[]
    for batch in df["batch"].unique():
        for concentration in df["concentration"].unique():
            df1=df[(df["batch"]==batch) & (df["concentration"]==concentration)]
            std=np.std(df1).to_list()[1:]
            std.insert(0,concentration)
            std.insert(0,batch)
            std_result.append(std)
    std_result=pd.DataFrame(std_result,columns=columns_name)
    return df,std_result

# Section 1: CuSO4

In [ ]:
%cd /home/nguyen/Desktop/Projects/Thay_Danh/smart-optical-sensor

In [10]:
# Feature extraction
indir="/home/nguyen/Desktop/Projects/Thay_Danh/smart-optical-sensor/data/regression/nano_Fe3+_v2"
outdir=indir
# Feature extraction
raw_image=os.path.join(indir,"raw_image")
os.system(f"python3 modules/feature_extraction_v2.py -i {raw_image}  -o {outdir}")

0

In [264]:
# EDA
# ROI
indir=os.path.join(outdir,"result","raw_roi","*.csv")
raw_roi_val,raw_roi_std=rgb(indir)
raw_roi_val.to_csv("data/regression/nano_Fe3+_v2/result/EDA/val_raw_roi.csv",index=False)
raw_roi_std.to_csv("data/regression/nano_Fe3+_v2/result/EDA/std_raw_roi.csv",index=False)
# Delta ROI
indir=os.path.join(outdir,"result","delta_normalized_roi","*.csv")
delta_roi_val,delta_roi_std=rgb(indir)
delta_roi_val.to_csv("data/regression/nano_Fe3+_v2/result/EDA/val_delta_roi.csv",index=False)
delta_roi_std.to_csv("data/regression/nano_Fe3+_v2/result/EDA/std_delta_roi.csv",index=False)
# Ratio ROI
indir=os.path.join(outdir,"result","ratio_normalized_roi","*.csv")
ratio_roi_val,ratio_roi_std=rgb(indir)
ratio_roi_val.to_csv("data/regression/nano_Fe3+_v2/result/EDA/val_ratio_roi.csv",index=False)
ratio_roi_std.to_csv("data/regression/nano_Fe3+_v2/result/EDA/std_ratio_roi.csv",index=False)

In [ ]:
def evaluation(learning_model,transform_model,x,y_real):
    X_t=transform_model.fit_transform(x)
    y_pred=learning_model.predict(X_t)
    # evaluation
    rmse   = np.sqrt(np.mean((y_real - y_pred)**2))
    mae    = np.mean(np.abs(y_real - y_pred))
    pmae   = np.mean(np.abs((y_real - y_pred) / y_real)) * 100
    r2     = r2_score(y_real,y_pred)
    return [rmse,mae,pmae,r2], y_pred

In [ ]:
# Train
# train will be the batch 1
# test will be on batch 2 and 3
def train_test(batch_train,df,degree_poly):    
    batch1=df[df["batch"]=="batch1"]
    batch2=df[df["batch"]=="batch2"]
    batch3=df[df["batch"]=="batch3"]
    # train
    train=df[df["batch"]==batch_train]
    x = train.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
    y = train["concentration"].values.astype(float)  # target variable
    poly = PolynomialFeatures(degree=degree_poly)
    X_t=poly.fit_transform(x)
    clf = LinearRegression()
    clf.fit(X_t, y)
    # test
    result_all=[]
    # batch 1
    x = batch1.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
    y = batch1["concentration"].values.astype(float)  # target variable
    result=evaluation(clf,poly,x,y)[0]
    result.insert(0,"batch1")
    result_all.append(result)
    # batch 2
    x = batch2.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
    y = batch2["concentration"].values.astype(float)  # target variable
    result=evaluation(clf,poly,x,y)[0]
    result.insert(0,"batch2")
    result_all.append(result)
    # batch 3
    x = batch3.drop(columns=["batch","concentration","image"]).values.astype(float) # 100 samples with 3 features each
    y = batch3["concentration"].values.astype(float)  # target variable
    result=evaluation(clf,poly,x,y)[0]
    result.insert(0,"batch3")
    result_all.append(result)
    result_all=pd.DataFrame(result_all,columns=["batch","rmse","mae","pmae","r^2"])
    return clf,poly,result_all,

ratio_roi_val

## Delta ROI result

In [238]:
delta_roi_val=delta_roi_val[["meanB","meanG","meanR","modeB","modeG","modeR","batch","concentration","image"]]

In [239]:
train_test("batch1",delta_roi_val,1)

(LinearRegression(),
 PolynomialFeatures(degree=1),
     batch      rmse       mae       pmae       r^2
 0  batch1  0.058274  0.042962   7.340340  0.954385
 1  batch2  0.087599  0.073077  16.378442  0.896925
 2  batch3  0.075556  0.056899   9.546373  0.923318)

In [240]:
train_test("batch2",delta_roi_val,1)

(LinearRegression(),
 PolynomialFeatures(degree=1),
     batch      rmse       mae       pmae       r^2
 0  batch1  0.111292  0.089834  21.443953  0.833624
 1  batch2  0.036098  0.029892   6.136283  0.982497
 2  batch3  0.066673  0.048665  15.026108  0.940289)

In [241]:
train_test("batch3",delta_roi_val,1)

(LinearRegression(),
 PolynomialFeatures(degree=1),
     batch      rmse       mae       pmae       r^2
 0  batch1  0.101116  0.069612  10.747692  0.862660
 1  batch2  0.096768  0.068562  21.461807  0.874218
 2  batch3  0.027038  0.023072   4.943581  0.990180)

## Ratio ROI result

In [242]:
ratio_roi_val=ratio_roi_val[["meanB","meanG","meanR","modeB","modeG","modeR","batch","concentration","image"]]

In [245]:
train_test("batch2",ratio_roi_val,1)

(LinearRegression(),
 PolynomialFeatures(degree=1),
     batch      rmse       mae      pmae       r^2
 0  batch1  0.040198  0.029811  5.150086  0.978295
 1  batch2  0.025489  0.020579  4.449209  0.991273
 2  batch3  0.025322  0.019104  4.298222  0.991387)

In [252]:
clf,transform,result=train_test("batch2",ratio_roi_val,1)

In [253]:
result

,batch,rmse,mae,pmae,r^2
0,batch1,0.040198,0.029811,5.150086,0.978295
1,batch2,0.025489,0.020579,4.449209,0.991273
2,batch3,0.025322,0.019104,4.298222,0.991387


In [258]:
y_predict=evaluation(clf,transform,ratio_roi_val[["meanB","meanG","meanR","modeB","modeG","modeR"]],ratio_roi_val["concentration"])[1]

In [259]:
y_predict=pd.DataFrame(y_predict,columns=["predicted_concentration"])

In [261]:
y_predict

,predicted_concentration
0,0.738646
1,0.256106
2,0.259688
3,0.276966
4,0.766292
...,...
109,0.761034
110,0.500290
111,0.958475
112,0.745874


In [263]:
result=pd.concat([ratio_roi_val,y_predict],axis=1)
result.to_csv("/home/nguyen/Desktop/Projects/Thay_Danh/smart-optical-sensor/data/balance_image/CuSO4_v2/result/EDA/result.csv",index=False)

In [ ]:
batch2

,meanB,meanG,meanR,modeB,modeG,modeR,batch,concentration,image
2,54.156631,51.649301,18.440206,54.658385,51.982316,19.292605,batch2,0.25,0.25-1_batch2.csv
3,55.083999,52.133319,18.664536,55.473839,52.933582,18.262313,batch2,0.25,0.25-6_batch2.csv
6,55.587422,45.125327,1.164099,57.148468,45.648692,0.000000,batch2,0.50,0.5-9_batch2.csv
8,55.455344,41.825299,0.235951,56.201933,42.124542,0.000000,batch2,0.75,0.75-6_batch2.csv
14,54.611141,41.074997,0.254196,55.485253,41.474654,0.000000,batch2,0.75,0.75-3_batch2.csv
15,54.773581,51.489504,17.841047,54.774397,52.601568,18.498660,batch2,0.25,0.25-10_batch2.csv
18,54.965783,40.774006,0.225387,55.698203,41.002278,0.000000,batch2,0.75,0.75-4_batch2.csv
21,55.589348,52.175838,18.958641,56.133712,52.397039,19.607843,batch2,0.25,0.25-5_batch2.csv
22,52.917449,38.153362,0.215029,53.482222,39.106525,0.000000,batch2,1.00,1.0-6_batch2.csv
26,55.313618,51.812465,18.144053,55.567383,52.483801,19.177321,batch2,0.25,0.25-7_batch2.csv


In [ ]:
train_test("batch3",ratio_roi_val,1)

(LinearRegression(),
 PolynomialFeatures(degree=1),
     batch      rmse       mae      pmae       r^2
 0  batch1  0.035759  0.026682  4.302148  0.982824
 1  batch2  0.040277  0.030620  7.971200  0.978209
 2  batch3  0.020918  0.016539  3.508918  0.994122)

## Raw data

In [ ]:
train_test("batch1",raw_roi_val,1)

(LinearRegression(),
 PolynomialFeatures(degree=1),
     batch      rmse       mae       pmae       r^2
 0  batch1  0.012076  0.008894   1.941747  0.998041
 1  batch2  0.097688  0.072320  22.075545  0.871813
 2  batch3  0.059295  0.046770   7.329123  0.952773)

In [ ]:
train_test("batch2",raw_roi_val,1)

(LinearRegression(),
 PolynomialFeatures(degree=1),
     batch      rmse       mae       pmae       r^2
 0  batch1  0.148779  0.103386  33.740636  0.702666
 1  batch2  0.014695  0.011660   2.579502  0.997099
 2  batch3  0.131415  0.092510  29.174668  0.768022)

In [ ]:
train_test("batch3",raw_roi_val,1)

(LinearRegression(),
 PolynomialFeatures(degree=1),
     batch      rmse       mae      pmae       r^2
 0  batch1  0.067458  0.051310  7.725923  0.938875
 1  batch2  0.058772  0.044610  8.079514  0.953601
 2  batch3  0.005013  0.004263  0.898775  0.999662)

In [ ]:
# raw_roi_val[raw_roi_val["batch"]=="batch1"].sort_values(["concentration","meanG","meanB","meanR"])
ratio_roi_val=ratio_roi_val[(ratio_roi_val["stdB"]<=2) & (ratio_roi_val["stdG"]<=2) & (ratio_roi_val["stdR"]<=2)].sort_values(["concentration","meanG","meanB","meanR"])